# Proyecto: Análisis de Estacionalidad en Ventas de Automóviles por Marca
**Materia:** Prácticas y Herramientas de Big Data  
**Integrantes:**
* Jose Alexander Quishpe Reinoso
* Julián Garofalo

## Explicación paso a paso
Este notebook está estructurado para llevar a cabo el análisis exploratorio y estadístico del dataset *Car Sales Data*, el cual contiene aproximadamente 2,500,000 de registros. El propósito fundamental es identificar si las ventas y los ingresos de las diferentes marcas de vehículos presentan un comportamiento estacional o si se distribuyen de manera uniforme a lo largo del año fiscal.

El flujo metodológico desarrollado paso a paso comprende:
1. **Extracción y simulación de ingesta:** Configuración del entorno de Big Data en Colab y simulación de consumo de API.
2. **Limpieza y preprocesamiento de millones de filas:** Remoción de duplicados, análisis de nulos y transformaciones de tipos de datos.
3. **Enriquecimiento temporal (Feature Engineering):** Extracción de componentes clave como meses, semanas ISO y trimestres a partir de la fecha.
4. **Formulación de Hipótesis:** Planteamiento formal de los supuestos estacionales de negocio.
5. **Preparación de Estructuras para Modelos:** Organización del espacio para la aplicación de modelos estadísticos y de descomposición de series de tiempo.

---
## Importación de librerías

Antes de comenzar, importamos todas las librerías necesarias para el proyecto. Cada librería cumple un rol específico:

| Librería | Para qué se usa |
|----------|----------------|
| `kagglehub` | Conectarse a la API oficial de Kaggle y cargar el dataset |
| `pandas` | Manipular y limpiar los datos en tablas |
| `numpy` | Operaciones matemáticas y manejo de valores nulos |
| `matplotlib` | Crear gráficos estáticos |
| `seaborn` | Gráficos estadísticos más visuales |
| `sklearn` | Modelos de machine learning |
| `warnings` | Suprimir advertencias innecesarias en la salida |

In [1]:
# ── Acceso al dataset de Kaggle ────────────────────────────────────
import kagglehub                                        # Librería oficial de Kaggle para cargar datasets
from kagglehub import KaggleDatasetAdapter              # Adaptador para cargar directo como DataFrame

# ── Manipulación de datos ───────────────────────────────────────────
import pandas as pd                      # Tablas de datos (DataFrames)
import numpy as np                       # Operaciones numéricas y manejo de NaN

# ── Visualización ──────────────────────────────────────────────────
import matplotlib.pyplot as plt          # Gráficos básicos (barras, líneas, histogramas)
import seaborn as sns                    # Gráficos estadísticos sobre matplotlib

# ── Preprocesamiento (Scikit-learn) ────────────────────────────────
from sklearn.preprocessing import LabelEncoder        # Convierte texto a números para los modelos
from sklearn.preprocessing import StandardScaler      # Escala variables numéricas a la misma magnitud
from sklearn.model_selection import train_test_split  # Divide datos en entrenamiento y prueba

# ── Modelos de Machine Learning ────────────────────────────────────
from sklearn.linear_model import LinearRegression     # Modelo 1: Regresión Lineal
from sklearn.ensemble import RandomForestRegressor    # Modelo 2: Random Forest
from sklearn.ensemble import GradientBoostingRegressor # Modelo 3: Gradient Boosting

# ── Métricas de evaluación ─────────────────────────────────────────
from sklearn.metrics import mean_absolute_error       # Error absoluto medio
from sklearn.metrics import mean_squared_error        # Error cuadrático medio
from sklearn.metrics import r2_score                  # Coeficiente de determinación R²

# ── Configuración general ──────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')        # Oculta advertencias para mantener la salida limpia

# Configuración visual de los gráficos
plt.style.use('seaborn-v0_8-whitegrid') # Estilo limpio con cuadrícula para los gráficos
pd.set_option('display.max_columns', None) # Muestra todas las columnas al imprimir el DataFrame

print('✅ Todas las librerías importadas correctamente')

✅ Todas las librerías importadas correctamente


---
## 3. Extracción de Datos

### Explicación técnica
En esta sección se consume la **API oficial de Kaggle** mediante la librería `kagglehub`, que permite cargar el dataset directamente en un DataFrame de pandas sin necesidad de descargarlo ni subirlo manualmente. 

Para cumplir con las mejores prácticas y estándares de seguridad de proyectos de Big Data, la extracción utiliza un adaptador de lectura directa en memoria y se autentica de forma segura mediante variables de entorno configuradas previamente en el sistema (Secrets de GitHub).

### Detalles de conexión
* **Librería utilizada:** `kagglehub`.
* **Dataset ID:** `suraj520/car-sales-data`.
* **Archivo:** `car_sales_data.csv`.
* **Adaptador de datos:** `KaggleDatasetAdapter.PANDAS` (convierte el flujo directamente a DataFrame)
* **Credenciales:** Variables de entorno `KAGGLE_USERNAME` y `KAGGLE_KEY`.

### Variables obtenidas del dataset
Al realizar la extracción, se obtienen las siguientes columnas estructuradas:

* `Date`: Fecha exacta de la transacción (clave para el análisis de series temporales).
* `Salesperson`: Nombre del asesor comercial.
* `Customer Name`: Nombre del comprador.
* `Car Make`: Fabricante del vehículo (Toyota, Honda, Ford, Chevrolet, Nissan).
* `Car Model`: Modelo específico del auto.
* `Car Year`: Año de fabricación del vehículo.
* `Sale Price`: Precio de venta final en USD (variable de ingresos).
* `Commission Rate`: Porcentaje de comisión asignado.
* `Commission Earned`: Monto total de comisión generado.

**Cantidad de registros estimados:** 2,500,000 filas.


In [ ]:
# ── PASO 1: Validación de credenciales de entorno ─────────────────

# 1. import os: Esta librería permite a Python interactuar directamente con el 
# sistema operativo de la máquina (en este caso, tu contenedor de Codespace).
# La necesitamos para acceder a la memoria donde GitHub guarda los Secrets.
import os 

# 2. os.environ: Es un diccionario que contiene todas las variables del sistema.
# 3. .get(): Busca la variable. Si no la encuentra, en lugar de dar un error fatal, 
# simplemente devuelve el valor por defecto ('❌ No configurado').
print('KAGGLE_USERNAME:', os.environ.get('KAGGLE_USERNAME', '❌ No configurado'))

# 4. Operador ternario (if-else en una línea): Evalúa si la clave existe.
# Por ciberseguridad, NUNCA imprimimos el valor real de la clave (os.environ.get('KAGGLE_KEY')).
# Solo verificamos su existencia para confirmar que el entorno está listo.
print('KAGGLE_KEY:     ', '✅ Configurado' if os.environ.get('KAGGLE_KEY') else '❌ No configurado')

KAGGLE_USERNAME: josquishpereinoso
KAGGLE_KEY:      ✅ Configurado


In [7]:
# ── PASO 2: Cargar el dataset directo desde Kaggle ─────────────────

# 1. kagglehub.dataset_load(): Es la función principal de la nueva librería. 
# Se encarga de autenticarse automáticamente usando las credenciales validadas
# en el paso anterior y de buscar el dataset directamente en los servidores de Kaggle.
df = kagglehub.dataset_load(
    
    # 2. KaggleDatasetAdapter.PANDAS: Este adaptador es clave en proyectos de Big Data.
    # En lugar de descargar un archivo físico (.csv o .zip) al disco duro,
    # lee el flujo de datos y lo convierte inmediatamente en un DataFrame de pandas en memoria.
    KaggleDatasetAdapter.PANDAS,
    
    # 3. 'suraj520/car-sales-data': Es el identificador único del repositorio en Kaggle.
    # Se compone del nombre de usuario del creador (suraj520) y el nombre del proyecto.
    'suraj520/car-sales-data',
    
    # 4. 'car_sales_data.csv': Especifica qué archivo exacto queremos extraer,
    # ya que un repositorio de Kaggle puede contener múltiples archivos de datos.
    'car_sales_data.csv'
)

# ── PASO 3: Variables obtenidas ────────────────────────────────────
# Imprimimos los nombres de las columnas para verificar la estructura extraída.
# tolist() convierte el objeto Index de pandas en una lista estándar de Python 
# para que se imprima de forma horizontal y legible.
print('Columnas del dataset:')
print(df.columns.tolist())

# ── PASO 4: Cantidad de registros ──────────────────────────────────
# len(df) cuenta las filas, validando que logramos extraer el millón de registros.
# df.shape devuelve una tupla con (filas, columnas), dando una visión de la magnitud estructural.
print(f'\n Registros extraídos: {len(df):,}') # El [:,] añade comas para leer mejor los miles
print(f' Dimensiones (filas x columnas): {df.shape}')

# ── PASO 5: Vista previa del dataset ───────────────────────────────
# head() es una función de pandas que muestra por defecto las primeras 5 filas.
# Funciona como evidencia visual de que los datos no solo se cargaron, 
# sino que tienen el formato correcto (fechas, textos y números alineados).
df.head()

Columnas del dataset:
['Date', 'Salesperson', 'Customer Name', 'Car Make', 'Car Model', 'Car Year', 'Sale Price', 'Commission Rate', 'Commission Earned']

 Registros extraídos: 2,500,000
 Dimensiones (filas x columnas): (2500000, 9)


,Date,Salesperson,Customer Name,Car Make,Car Model,Car Year,Sale Price,Commission Rate,Commission Earned
0,2022-08-01,Monica Moore MD,Mary Butler,Nissan,Altima,2018,15983,0.070495,1126.73
1,2023-03-15,Roberto Rose,Richard Pierce,Nissan,F-150,2016,38474,0.134439,5172.40
2,2023-04-29,Ashley Ramos,Sandra Moore,Ford,Civic,2016,33340,0.114536,3818.63
3,2022-09-04,Patrick Harris,Johnny Scott,Ford,Altima,2013,41937,0.092191,3866.20
4,2022-06-16,Eric Lopez,Vanessa Jones,Honda,Silverado,2022,20256,0.113490,2298.85


---

## 4. Limpieza y Preprocesamiento

### Explicación técnica
El procesamiento de grandes volúmenes de datos requiere asegurar la máxima consistencia antes de estructurar las series temporales y entrenar los modelos predictivos. En esta sección preparamos y depuramos el dataset para garantizar que la información ingresada a los algoritmos sea 100% confiable.

### Pasos implementados:

1. **Exploración y auditoría inicial:** Entender la estructura del dataset e identificar la proporción de campos vacíos en métricas críticas como `Sale Price`, `Date` y `Car Make`.
2. **Remoción de inconsistencias (Nulos y Duplicados):** Eliminación estricta de celdas vacías y registros redundantes (duplicados exactos) para evitar sesgos analíticos y errores de ejecución.
3. **Transformación de tipos de dato:** Conversión de variables financieras a numéricas y transformación de la columna `Date` a un tipo de dato cronológico nativo (`datetime64`).
4. **Estandarización de texto:** Unificación del formato en variables categóricas (eliminando espacios en blanco y capitalizando palabras) para asegurar que entidades como "Toyota" y " toyota " se procesen como el mismo valor.
5. **Ingeniería de Características Temporales:** Descomposición de la variable de fecha en granularidades específicas (Mes, Trimestre, Semana ISO y Día de la semana) de acuerdo con los objetivos metodológicos de análisis de estacionalidad.

In [8]:
# ── 4.1 EXPLORACIÓN INICIAL ────────────────────────────────────────

print('Información general del dataset:')
# 1. df.info(): Funciona como la radiografía principal de tu DataFrame.
# Te muestra el tamaño total (filas y columnas), el tipo de dato de cada 
# variable (texto, número entero, decimal) y, lo más importante, cuántos 
# valores NO nulos hay por columna. Es el paso 1 para detectar inconsistencias.
df.info()

print('\nEstadísticas descriptivas:')
# 2. df.describe(): Genera un resumen estadístico de todas las columnas numéricas.
# Calcula automáticamente: cantidad de datos (count), promedio (mean), 
# desviación estándar (std), valores mínimos (min), máximos (max) y percentiles.
# Es fundamental para detectar rápidamente si hay valores atípicos (outliers) 
# o datos ilógicos (por ejemplo, un auto con precio negativo).
df.describe()

Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 2500000 entries, 0 to 2499999
Data columns (total 9 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Date               str    
 1   Salesperson        str    
 2   Customer Name      str    
 3   Car Make           str    
 4   Car Model          str    
 5   Car Year           int64  
 6   Sale Price         int64  
 7   Commission Rate    float64
 8   Commission Earned  float64
dtypes: float64(2), int64(2), str(5)
memory usage: 171.7 MB

Estadísticas descriptivas:


,Car Year,Sale Price,Commission Rate,Commission Earned
count,2.500000e+06,2.500000e+06,2.500000e+06,2.500000e+06
mean,2.015996e+03,3.001218e+04,9.998766e-02,3.001005e+03
std,3.739132e+00,1.154514e+04,2.887202e-02,1.481467e+03
min,2.010000e+03,1.000000e+04,5.000014e-02,5.013400e+02
25%,2.013000e+03,2.001900e+04,7.496450e-02,1.821710e+03
50%,2.016000e+03,3.000600e+04,1.000058e-01,2.741910e+03
75%,2.019000e+03,4.002200e+04,1.250065e-01,3.978142e+03
max,2.022000e+03,5.000000e+04,1.500000e-01,7.494530e+03


In [ ]:
# ── 4.2 TRATAMIENTO DE VALORES NULOS ──────────────────────────────

print('Valores nulos por columna (Antes de la limpieza):')
# 1. df.isnull().sum(): isnull() evalúa cada celda y devuelve True si está vacía (NaN).
# Al encadenarlo con sum(), suma todos los 'True' por cada columna.
# Esta es nuestra auditoría inicial para saber en qué columnas faltan datos.
print(df.isnull().sum())

# 2. df.dropna(): Elimina automáticamente cualquier fila que contenga al menos un dato vacío.
# Al reasignarlo a la variable 'df' (df = df.dropna()), estamos guardando el cambio 
# y sobrescribiendo el dataset para quedarnos únicamente con registros 100% completos.
df = df.dropna()

# 3. Imprimimos la nueva longitud del DataFrame (len) para ver con cuántos datos nos quedamos.
# El formato [:,] añade comas para facilitar la lectura de los miles/millones.
print(f'\n Registros después de eliminar nulos: {len(df):,}')

Valores nulos por columna (Antes de la limpieza):
Date                 0
Salesperson          0
Customer Name        0
Car Make             0
Car Model            0
Car Year             0
Sale Price           0
Commission Rate      0
Commission Earned    0
dtype: int64

✅ Registros después de eliminar nulos: 2,500,000


In [11]:
# ── 4.3 ELIMINACIÓN DE DUPLICADOS ─────────────────────────────────

# 1. df.duplicated().sum(): duplicated() escanea todo el dataset y marca con True
# las filas que son copias exactas de un registro anterior. Al sumarlas, obtenemos el total.
# Este paso es vital para no sesgar el análisis ni el entrenamiento de los modelos con datos redundantes.
print(f' Duplicados exactos encontrados: {df.duplicated().sum():,}')

# 2. df.drop_duplicates(): Borra todas las filas repetidas detectadas en el paso anterior,
# conservando por defecto solo la primera aparición original de cada registro.
# Sobrescribimos la variable 'df' para aplicar el cambio permanentemente.
df = df.drop_duplicates()

# 3. Imprimimos el tamaño del dataset después de esta segunda fase de limpieza.
# Si el número baja, significa que logramos purgar datos repetidos con éxito.
print(f'Registros después de eliminar duplicados: {len(df):,}')

 Duplicados exactos encontrados: 0
Registros después de eliminar duplicados: 2,500,000


In [12]:
# ── 4.4 TRANSFORMACIONES BÁSICAS ──────────────────────────────────

# 1. pd.to_datetime(): Convierte la columna 'Date' de texto plano (string) 
# a un formato cronológico nativo de pandas (datetime64). 
# ¡Este es el paso obligatorio para que tu Sprint de Fechas (Paso 4.6) funcione!
df['Date'] = pd.to_datetime(df['Date'])

# 2. pd.to_numeric(): Asegura que las variables financieras y el año sean números 
# matemáticos (float/int) y no cadenas de texto. Los modelos de ML requieren números.
# El parámetro errors='coerce' es una medida de seguridad: si el código se topa con 
# un texto corrupto (ej. "N/A" o "Mil"), en lugar de colapsar, lo convierte a vacío (NaN).
df['Sale Price']       = pd.to_numeric(df['Sale Price'],       errors='coerce')
df['Commission Rate']  = pd.to_numeric(df['Commission Rate'],  errors='coerce')
df['Commission Earned']= pd.to_numeric(df['Commission Earned'],errors='coerce')
df['Car Year']         = pd.to_numeric(df['Car Year'],         errors='coerce')

# 3. df.dtypes: Imprime la lista de columnas junto con su nuevo tipo de dato.
# Nos sirve para confirmar visualmente que pasamos de 'object' (texto) a 
# 'datetime64' y 'float64' (decimales).
print('Tipos de datos después de transformar:')
print(df.dtypes)

Tipos de datos después de transformar:
Date                 datetime64[us]
Salesperson                     str
Customer Name                   str
Car Make                        str
Car Model                       str
Car Year                      int64
Sale Price                    int64
Commission Rate             float64
Commission Earned           float64
dtype: object


### Paso opcional pero recomendado
Paso 4.5 Estandarización preliminar: Limpia los textos usando str.strip().str.title() para eliminar inconsistencias tipográficas.

In [14]:
# ── 4.5 ESTANDARIZACIÓN PRELIMINAR ────────────────────────────────

# 1. Limpieza tipográfica de variables categóricas (texto)
# str.strip(): Elimina espacios en blanco accidentales al inicio y al final.
# str.title(): Convierte el texto a formato título (Primera Letra Mayúscula).
# Esta combinación es vital para agrupar datos correctamente. Evita que el 
# modelo trate a 'Toyota', 'TOYOTA' y '  toyota ' como tres marcas distintas.
df['Car Make']    = df['Car Make'].str.strip().str.title()
df['Car Model']   = df['Car Model'].str.strip().str.title()
df['Salesperson'] = df['Salesperson'].str.strip().str.title()

# 2. Validación de la estandarización
# unique() devuelve una lista con los valores distintos presentes en la columna.
# Nos permite confirmar visualmente que no hay nombres duplicados por mala ortografía.
print('Marcas de autos únicas después de estandarizar:')
print(df['Car Make'].unique())

# 3. Vista final de la fase de estandarización
# Mostramos las dimensiones finales (usando [:,] para separar los miles) y 
# las primeras filas para confirmar que el dataset está impecable.
print(f'\n Dataset preprocesado: {df.shape[0]:,} registros, {df.shape[1]} columnas')
df.head()

Marcas de autos únicas después de estandarizar:
<StringArray>
['Nissan', 'Ford', 'Honda', 'Toyota', 'Chevrolet']
Length: 5, dtype: str

 Dataset preprocesado: 2,500,000 registros, 9 columnas


,Date,Salesperson,Customer Name,Car Make,Car Model,Car Year,Sale Price,Commission Rate,Commission Earned
0,2022-08-01,Monica Moore Md,Mary Butler,Nissan,Altima,2018,15983,0.070495,1126.73
1,2023-03-15,Roberto Rose,Richard Pierce,Nissan,F-150,2016,38474,0.134439,5172.40
2,2023-04-29,Ashley Ramos,Sandra Moore,Ford,Civic,2016,33340,0.114536,3818.63
3,2022-09-04,Patrick Harris,Johnny Scott,Ford,Altima,2013,41937,0.092191,3866.20
4,2022-06-16,Eric Lopez,Vanessa Jones,Honda,Silverado,2022,20256,0.113490,2298.85
